# 华泰金工全频段量价选股模型复现（LightGBM版）

## 项目框架搭建
本notebook演示从数据获取到回测的完整流程，使用沪深300成分股分钟数据计算两个简单因子，训练LightGBM模型。

### 注意事项
1. 确保已配置RQData用户名密码（在VeighNa Station中设置）
2. 数据下载可能需要较长时间，建议先使用少量股票测试
3. 因子计算部分需要根据研报实现27个高频因子，此处仅示例两个

**MFML_v2 更新**：
- 实现日频因子计算（每日分钟收益率均值和波动率）
- 使用10日滞后特征作为模型输入（20维特征向量）
- 标签为未来10日收益率

In [ ]:
# ============================================================================
# Windows 多进程问题修复
# ============================================================================
import multiprocessing
import sys
import warnings
warnings.filterwarnings('ignore')

# Windows 多进程配置
if sys.platform == 'win32':
    try:
        multiprocessing.set_start_method('spawn')
        print('多进程启动方法设置为 spawn (Windows兼容性)')
    except RuntimeError:
        pass  # 已经设置过

In [ ]:
# 1. 导入必要的库
# vnpy相关
from vnpy.trader.setting import SETTINGS
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import  HistoryRequest
from vnpy.trader.datafeed import get_datafeed
from vnpy.alpha.lab import AlphaLab
from vnpy.alpha.dataset import AlphaDataset
from vnpy.alpha.dataset.processor import (
    process_drop_na, process_fill_na, process_cs_norm
)

# 数据处理
import polars as pl

print('导入完成')

In [ ]:
# 2. 配置RQData数据服务

# 检查是否已配置用户名密码
print('当前数据服务配置:')
print('datafeed.username:', SETTINGS.get('datafeed.username', '未设置'))
print('datafeed.password:', SETTINGS.get('datafeed.password', '未设置'))

# 初始化数据服务
datafeed = get_datafeed()
print(f'数据服务类型: {datafeed.__class__.__name__}')

# 尝试初始化
inited = datafeed.init(output=print)
print(f'初始化结果: {inited}')


In [ ]:
# 3. 创建AlphaLab实验室
import os
lab_path = os.path.join(os.getcwd(), 'lab2')
lab = AlphaLab(lab_path)
print(f'AlphaLab路径: {lab_path}')

In [ ]:
# 4. 数据下载函数（修复时区问题）
def download_bars(symbol, exchange, start, end, interval, tz='Asia/Shanghai'):
    """
    下载K线数据并修复时区问题
    """
    from zoneinfo import ZoneInfo
    tz_sh = ZoneInfo(tz)
    start_dt = start.replace(tzinfo=tz_sh) if start.tzinfo is None else start.astimezone(tz_sh)
    end_dt = end.replace(tzinfo=tz_sh) if end.tzinfo is None else end.astimezone(tz_sh)
    
    req = HistoryRequest(
        symbol=symbol,
        exchange=exchange,
        start=start_dt,
        end=end_dt,
        interval=interval
    )
    
    bars = datafeed.query_bar_history(req, output=print)
    if bars:
        # 转换时区相关datetime为时区无关（本地时间）
        for bar in bars:
            if bar.datetime.tzinfo is not None:
                bar.datetime = bar.datetime.astimezone(tz_sh).replace(tzinfo=None)
    return bars

print('数据下载函数定义完成')

In [ ]:
# 5. 下载数据
index_symbol = "000300.SSE"
rq_index_symbol = "000300.XSHG"
extended_days: int = 0
# 时间范围：2023-01-01 至 2026-03-27
from datetime import datetime
test_start = datetime(2025, 1, 1)
test_end = datetime(2026, 3, 27)
start = datetime(2013, 1, 1)
end = datetime(2026, 3, 27)
interval = Interval.MINUTE                  #数据频率

# import rqdatac as rq
# # 下载指数成分股 时间对应的成分股列表
# data = rq.index_components(rq_index_symbol, start_date=start, end_date=end)
# # 转换合约代码
# index_components = {}
# for dt, rq_symbols in data.items():
#     vt_symbols: list = []
#
#     for rq_symbol in rq_symbols:
#         vt_symbol = rq_symbol.replace("XSHG", "SSE").replace("XSHE", "SZSE")
#         vt_symbols.append(vt_symbol)
#
#     index_components[dt.strftime("%Y-%m-%d")] = vt_symbols    #index_components = {"%Y-%m-%d":[成分股列表]，....} vnpy代码格式
#
#
# # 保存到数据中心
# lab.save_component_data(index_symbol, index_components)
# 加载指数成分股代码
component_symbols = lab.load_component_symbols(index_symbol, start, end)
print(component_symbols)

In [ ]:
# 添加回测参数配置
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5/10000,
        short_rate=10/10000,
        size=1,
        pricetick=0.0001,
    )

In [ ]:
# 转换时间格式
from vnpy.trader.database import DB_TZ
from vnpy.alpha import  logger
from tqdm import tqdm

start = start.replace(tzinfo=DB_TZ) #下载的时候要有时区  下下来后没有时区了
end = end.replace(tzinfo=DB_TZ)

# # 除了成分股，还要下载指数数据
# task_symbols = component_symbols
#
# # 遍历下载数据
#
#
from tqdm import tqdm
#
#
# for vt_symbol in tqdm(task_symbols):
#     symbol, exchange_str = vt_symbol.split(".")
#     print(f'下载 {vt_symbol} ...')
#     req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval)
#     bars = datafeed.query_bar_history(req)
#
#     if bars:
#         lab.save_bar_data(bars)
#         print(f'  -> 保存 {len(bars)} 条K线')
#     else:
#         logger.error(f"下载{vt_symbol}数据失败")

In [ ]:
# 6. 加载数据
start = start.replace(tzinfo=None)
end = end.replace(tzinfo=None)
df = lab.load_bar_df(
    vt_symbols=["600277.SSE"],
    interval=Interval.DAILY,
    start=start,
    end=end,
    extended_days=0
)

if df is not None:
    print(f'数据形状: {df.shape}')
    print(df.tail())
else:
    print('加载数据失败')

In [ ]:
print(df['vt_symbol'].unique())

In [ ]:
task_symbols = list(set(component_symbols) - set(df['vt_symbol'].unique().to_list()))
print(len(task_symbols))

In [ ]:
start = start.replace(tzinfo=DB_TZ)
end = end.replace(tzinfo=DB_TZ)
n = 0
for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval)
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'{n}只股票')
        print(f'  -> 保存 {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

In [ ]:
# # 加载回测所需daily数据
# test_start = test_start.replace(tzinfo=DB_TZ)
# test_end = test_end.replace(tzinfo=DB_TZ)
# task_symbols = component_symbols + [index_symbol]
# n = 0
# for vt_symbol in tqdm(task_symbols):
#     symbol, exchange_str = vt_symbol.split(".")
#     print(f'下载 {vt_symbol} ...')
#     req = HistoryRequest(symbol, Exchange(exchange_str), test_start, test_end, Interval.DAILY)
#     bars = datafeed.query_bar_history(req)
#
#     if bars:
#         n += 1
#         lab.save_bar_data(bars)
#         print(f'{n}只股票')
#         print(f'  -> 保存 {len(bars)} 条K线')
#     else:
#         logger.error(f"下载{vt_symbol}数据失败")

In [ ]:
import rqdatac as rq
quota = rq.user.get_quota()
print(quota)

## 7. 日频因子计算

从分钟数据计算日频因子：
- **mean_ret**: 每日分钟收益率的均值
- **vol_ret**: 每日分钟收益率的波动率（标准差）

然后将因子转换为10日滞后特征作为模型输入。

In [ ]:
# 7.1 数据准备

# 确保数据按股票和时间排序
df_sorted = df.sort(['vt_symbol', 'datetime'])

# 提取日期和时间用于分组
df_sorted = df_sorted.with_columns([
    pl.col('datetime').dt.date().alias('date'),
    pl.col('datetime').dt.time().alias('time')
])

# 计算分钟收益率
df_sorted = df_sorted.with_columns([
    ((pl.col('close') / pl.col('close').shift(1).over(['vt_symbol', 'date'])) - 1).alias('min_ret')
])

# 计算振幅
df_sorted = df_sorted.with_columns([
    ((pl.col('high') - pl.col('low')) / pl.col('low')).alias('amplitude')
])

# 定义半小时时间段
def get_time_bucket_expr(time_col):
    """将时间映射到半小时桶"""
    return (
        pl.when((time_col >= pl.time(9, 30)) & (time_col < pl.time(10, 0))).then(1)
        .when((time_col >= pl.time(10, 0)) & (time_col < pl.time(10, 30))).then(2)
        .when((time_col >= pl.time(10, 30)) & (time_col < pl.time(11, 0))).then(3)
        .when((time_col >= pl.time(11, 0)) & (time_col < pl.time(11, 30))).then(4)
        .when((time_col >= pl.time(13, 0)) & (time_col < pl.time(13, 30))).then(5)
        .when((time_col >= pl.time(13, 30)) & (time_col < pl.time(14, 0))).then(6)
        .when((time_col >= pl.time(14, 0)) & (time_col < pl.time(14, 30))).then(7)
        .when((time_col >= pl.time(14, 30)) & (time_col <= pl.time(15, 0))).then(8)
        .otherwise(None)
        .alias('time_bucket')
    )

df_sorted = df_sorted.with_columns([get_time_bucket_expr(pl.col('time'))])

factor = []  # 存储所有因子名字

print('数据准备完成')
print(f'数据形状: {df_sorted.shape}')
print(df_sorted)

In [ ]:
# 7.2 计算因子
# 因子1: late_skew_ret (尾盘收益率偏度, 14:30-15:00)
# 偏度 = E[(X-μ)^3] / σ^3

late_data = df_sorted.filter(pl.col('time_bucket') == 8)

# 计算偏度
late_skew = late_data.group_by(['vt_symbol', 'date']).agg([
    pl.col('min_ret').mean().alias('mean_ret'),
    pl.col('min_ret').std().alias('std_ret'),
    ((pl.col('min_ret') - pl.col('min_ret').mean())**3).mean().alias('moment3')
]).with_columns([
    (pl.col('moment3') / (pl.col('std_ret')**3 + 1e-10)).alias('data'),
    pl.col('date').cast(pl.Datetime).alias('datetime')
]).select(['datetime', 'vt_symbol', 'data'])

# 处理缺失值
late_skew = late_skew.with_columns([
    pl.col('data').fill_nan(0).fill_null(0)
])

factor.append(('late_skew_ret', late_skew))
print(f'late_skew_ret: {late_skew.shape[0]} 条记录')
print(late_skew.head())

In [ ]:
# 因子2: down_vol_perc (下行收益率波动占比)
# 负收益率分钟的波动占全天总波动的比例

# 先计算每日的全局均值和标准差
daily_ret_stats = df_sorted.group_by(['vt_symbol', 'date']).agg([
    pl.col('min_ret').mean().alias('daily_mean_ret'),
    pl.col('min_ret').std().alias('daily_std_ret')
])

# 下行波动（负收益率的方差）
down_vol_data = df_sorted.filter(pl.col('min_ret') < 0)
down_vol = down_vol_data.group_by(['vt_symbol', 'date']).agg([
    ((pl.col('min_ret') - pl.col('min_ret').mean())**2).mean().alias('down_var')
])

# 合并并计算比例
down_vol_perc = daily_ret_stats.join(down_vol, on=['vt_symbol', 'date'], how='left')
down_vol_perc = down_vol_perc.with_columns([
    (pl.col('down_var') / (pl.col('daily_std_ret')**2 + 1e-10)).fill_nan(0).fill_null(0).alias('data')
]).with_columns([
    pl.col('date').cast(pl.Datetime).alias('datetime')
]).select(['datetime', 'vt_symbol', 'data'])

factor.append(('down_vol_perc', down_vol_perc))
print(f'down_vol_perc: {down_vol_perc.shape[0]} 条记录')
print(down_vol_perc.head())

In [ ]:
# 因子3: corr_ret_lastret (前后两分钟收益率相关性)
# 相邻分钟收益率的一阶自相关性

df_with_lag = df_sorted.with_columns([
    pl.col('min_ret').shift(1).over(['vt_symbol', 'date']).alias('min_ret_lag1')
])
# 过滤掉跨日的数据（lag为null的行）
df_with_lag = df_with_lag.filter(pl.col('min_ret_lag1').is_not_null())

corr_ret_lastret = df_with_lag.group_by(['vt_symbol', 'date']).agg([
    pl.corr('min_ret', 'min_ret_lag1').alias('data')
]).with_columns([
    pl.col('date').cast(pl.Datetime).alias('datetime')
]).select(['datetime', 'vt_symbol', 'data'])

# 处理缺失值
corr_ret_lastret = corr_ret_lastret.with_columns([
    pl.col('data').fill_nan(0).fill_null(0)
])

factor.append(('corr_ret_lastret', corr_ret_lastret))
print(f'corr_ret_lastret: {corr_ret_lastret.shape[0]} 条记录')
print(corr_ret_lastret.head())

In [ ]:
# 因子4: corr_close_nextopen (前一分钟收盘价与后一分钟开盘价相关性)

df_with_open_lag = df_sorted.with_columns([
    pl.col('close').shift(1).over(['vt_symbol', 'date']).alias('close_lag1')
])
df_with_open_lag = df_with_open_lag.filter(pl.col('close_lag1').is_not_null())

corr_close_nextopen = df_with_open_lag.group_by(['vt_symbol', 'date']).agg([
    pl.corr('close_lag1', 'open').alias('data')
]).with_columns([
    pl.col('date').cast(pl.Datetime).alias('datetime')
]).select(['datetime', 'vt_symbol', 'data'])

# 处理缺失值
corr_close_nextopen = corr_close_nextopen.with_columns([
    pl.col('data').fill_nan(0).fill_null(0)
])

factor.append(('corr_close_nextopen', corr_close_nextopen))
print(f'corr_close_nextopen: {corr_close_nextopen.shape[0]} 条记录')
print(corr_close_nextopen.head())

In [ ]:
# 因子5-10: volume_perc2 到 volume_perc7 (各半小时成交量占比)

# 计算全天总成交量
daily_total_vol = df_sorted.group_by(['vt_symbol', 'date']).agg([
    pl.col('volume').sum().alias('total_volume')
])

# 各时段成交量占比
for bucket in range(2, 8):  # 2-7
    bucket_vol = df_sorted.filter(pl.col('time_bucket') == bucket).group_by(['vt_symbol', 'date']).agg([
        pl.col('volume').sum().alias('bucket_volume')
    ])
    
    vol_perc = daily_total_vol.join(bucket_vol, on=['vt_symbol', 'date'], how='left')
    vol_perc = vol_perc.with_columns([
        (pl.col('bucket_volume') / (pl.col('total_volume') + 1e-10)).fill_nan(0).fill_null(0).alias('data')
    ]).with_columns([
        pl.col('date').cast(pl.Datetime).alias('datetime')
    ]).select(['datetime', 'vt_symbol', 'data'])
    
    factor.append((f'volume_perc{bucket}', vol_perc))
    print(f'volume_perc{bucket}: {vol_perc.shape[0]} 条记录')
    print(vol_perc.head())

print('因子5-10计算完成')

In [ ]:
# 因子11: early_corr_volume_ret (早盘成交量与收益率相关性, 9:30-10:30)
# 早盘时段: time_bucket 1 (9:30-10:00) 和 2 (10:00-10:30)

early_data = df_sorted.filter(pl.col('time_bucket').is_in([1, 2]))
early_corr_volume_ret = early_data.group_by(['vt_symbol', 'date']).agg([
    pl.corr('min_ret', 'volume').alias('data')
]).with_columns([
    pl.col('date').cast(pl.Datetime).alias('datetime')
]).select(['datetime', 'vt_symbol', 'data'])

# 处理缺失值
early_corr_volume_ret = early_corr_volume_ret.with_columns([
    pl.col('data').fill_nan(0).fill_null(0)
])

factor.append(('early_corr_volume_ret', early_corr_volume_ret))
print(f'early_corr_volume_ret: {early_corr_volume_ret.shape[0]} 条记录')
print(early_corr_volume_ret.head())

In [ ]:
# 因子12: corr_volume_amplitude (成交量与振幅相关性)

corr_volume_amplitude = df_sorted.group_by(['vt_symbol', 'date']).agg([
    pl.corr('amplitude', 'volume').alias('data')
]).with_columns([
    pl.col('date').cast(pl.Datetime).alias('datetime')
]).select(['datetime', 'vt_symbol', 'data'])

# 处理缺失值
corr_volume_amplitude = corr_volume_amplitude.with_columns([
    pl.col('data').fill_nan(0).fill_null(0)
])

factor.append(('corr_volume_amplitude', corr_volume_amplitude))
print(f'corr_volume_amplitude: {corr_volume_amplitude.shape[0]} 条记录')
print(corr_volume_amplitude.head())

In [ ]:
# 验证所有因子

print(f'共计算 {len(factor)} 个因子:')
for name, fdf in factor:
    print(f'  {name}: {fdf.shape[0]} 条记录')

# 显示所有因子名称
print('\n因子列表:')
for i, (name, _) in enumerate(factor, 1):
    print(f'  {i}. {name}')

In [ ]:
# 创建10日滞后特征（12个因子 × 10天 = 120个特征）

# 每个因子的滞后特征作为独立的add_feature输入

# 创建滞后特征DataFrame列表
all_lag_dfs = []

for factor_name, factor_df in factor:
    # 先按股票和日期排序
    factor_df = factor_df.sort(['vt_symbol', 'datetime'])
    
    for lag in range(1, 11):  # lag 1到10
        # 创建滞后特征
        lag_df = factor_df.with_columns([
            pl.col('data').shift(lag).over('vt_symbol').alias('data')
        ])
        all_lag_dfs.append((f'{factor_name}_lag_{lag}', lag_df))

print('滞后特征创建完成')
print(f'共创建 {len(all_lag_dfs)} 个滞后特征 (12因子 x 10天)')
print(f'特征列表: {[name for name, _ in all_lag_dfs][:5]}... (显示前5个)')

In [ ]:
# 7.3 准备基础数据（日频）

# 获取日频的close价格用于计算标签
# 从原始分钟数据中提取日频close
daily_close = df_sorted.group_by(['vt_symbol', 'date']).agg([
    pl.col('close').last().alias('close')
])

# 转换为datetime
daily_close = daily_close.with_columns([
    pl.col('date').cast(pl.Datetime).alias('datetime')
]).select(['datetime', 'vt_symbol', 'close'])

# 排序
daily_close = daily_close.sort(['vt_symbol', 'datetime'])

print('日频基础数据准备完成')
print(f'daily_close形状: {daily_close.shape}')
print()
print('daily_close示例:')
print(daily_close.head(10))

In [ ]:
# 8. 创建AlphaDataset并添加特征

# 划分训练集、验证集、测试集（日频）
train_start = '2023-01-01'
train_end = '2023-12-31'
valid_start = '2024-01-01'
valid_end = '2024-12-31'
test_start = '2025-01-01'
test_end = '2026-03-27'

# 使用日频close数据作为基础数据
dataset = AlphaDataset(
    df=daily_close,
    train_period=(train_start, train_end),
    valid_period=(valid_start, valid_end),
    test_period=(test_start, test_end),
    process_type='append'
)

# 添加120个滞后特征（12因子 × 10天）
for name, lag_df in all_lag_dfs:
    dataset.add_feature(name, result=lag_df)

# 设置标签：未来10日收益率
# 使用 ts_delay(close, -10) / close - 1 计算未来10日收益
dataset.set_label('(ts_delay(close, -10) / close) - 1')

print('数据集创建完成')
print(f'特征数量: 120 (12因子 × 10天滞后)')
print(f'标签: 未来10日收益率')

In [ ]:
# 9. 添加数据预处理器

from functools import partial

# 删除标签缺失的行（最后10条数据没有未来10日收益）
dataset.add_processor('learn', partial(process_drop_na, names=['label']))

# 对标签进行截面z-score标准化（使标签分布更稳定）
dataset.add_processor('learn', partial(process_cs_norm, names=['label'], method='zscore'))

# 对特征进行缺失值填充（用0填充）
# 注意：前10天滞后特征会有NA，这是正常的
dataset.add_processor('infer', partial(process_fill_na, fill_value=0.0, fill_label=False))

print('预处理器添加完成')

In [ ]:
# 10. 准备和处理数据

# 准备数据（计算特征和标签，合并到result_df）
dataset.prepare_data()

# 处理数据（应用预处理器）
dataset.process_data()

print('数据准备和处理完成')
print()
print(f'raw_df列: {dataset.raw_df.columns}')
print()
print(f'learn_df形状: {dataset.learn_df.shape}')

In [ ]:
# 11. 保存数据集

name = 'daily_10d_lgb'
lab.save_dataset(name, dataset)

print(f'数据集已保存: {name}')

In [ ]:
# 12. 加载数据集并训练模型

import numpy as np
from vnpy.alpha import Segment, AlphaDataset, AlphaModel
from vnpy.alpha.model.models.lgb_model import LgbModel

# 从文件缓存加载
dataset: AlphaDataset = lab.load_dataset(name)

# 创建模型对象
model: AlphaModel = LgbModel(seed=42)

print('模型和数据集准备完成，开始训练...')

In [ ]:
# 13. 训练模型

model.fit(dataset)

print('模型训练完成')

In [ ]:
# 14. 预测和评估

# 在测试集上预测
predictions = model.predict(dataset, Segment.TEST)

print(f'预测完成，预测样本数: {len(predictions)}')
print(f'预测值范围: [{predictions.min():.6f}, {predictions.max():.6f}]')
print(f'预测值均值: {predictions.mean():.6f}')

In [ ]:
# 15. 查看模型特征重要性

model.detail()

In [ ]:
# 保存模型
lab.save_model(name, model)

In [ ]:
model: AlphaModel = lab.load_model(name)

In [ ]:
# 用模型在测试集上预测
pre: np.ndarray = model.predict(dataset, Segment.TEST)

# 加载测试集数据
df_t: pl.DataFrame = dataset.fetch_infer(Segment.TEST)

# 合并预测信号列
df_t = df_t.with_columns(pl.Series(pre).alias("signal"))

# 提取信号数据
signal: pl.DataFrame = df_t["datetime", "vt_symbol", "signal"]
print(df_t)
print(signal)

In [ ]:
# 保存信号数据
lab.save_signal(name, signal)

In [ ]:
# 加载模块
import importlib
from datetime import datetime

from vnpy.alpha.strategy import BacktestingEngine

import vnpy.alpha.strategy.strategies.equity_demo_strategy as equity_demo_strategy

In [ ]:
# 重载策略类
importlib.reload(equity_demo_strategy)
EquityDemoStrategy = equity_demo_strategy.EquityDemoStrategy# 重载策略类

In [ ]:
# 从文件加载信号数据
signal = lab.load_signal(name)
print(signal)

In [ ]:
# 创建回测引擎对象
engine = BacktestingEngine(lab)

# 设置回测参数
engine.set_parameters(
    vt_symbols=component_symbols,
    interval=Interval.DAILY,
    start=datetime(2025, 1, 1),
    end=datetime(2026, 3, 27),
    capital=100000000
)

# 添加策略实例
setting = {"top_k": 10, "n_drop":10, "min_days":5}
engine.add_strategy(EquityDemoStrategy, setting, signal)

In [ ]:
# 执行回测任务
engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()


In [ ]:
print(engine.trades)

In [ ]:
# 显示超额收益分析结果
index_symbol: str = "000300.SSE"
engine.show_performance(benchmark_symbol=index_symbol)